# Temperature correction for real-data time-lapse resistivity

This notebook builds a sensor-interpolated temperature field on the ERT inversion mesh and applies the WRR-style temperature correction to the ADTLERT inverted resistivity models.

The correction uses `Tmean = 6.5 deg C` and `ct = 2.0 percent / deg C`. This is a practical first-pass correction from the available DTP/TEROS sensor data, not a full reproduction of the paper's thermal diffusion model.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import re

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import LogNorm, TwoSlopeNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "axes.labelsize": 13,
        "axes.titlesize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 10,
        "figure.dpi": 120,
    }
)
np.set_printoptions(precision=4, suppress=True)

In [ ]:
# Resolve paths and main controls.
here = Path.cwd().resolve()
candidates = [here, *here.parents]
project_root = next(
    (
        p
        for p in candidates
        if (p / "adtlert").exists() and (p / "ProcessedData").exists()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Cannot locate project root containing adtlert/ and ProcessedData/."
    )

data_dir = project_root / "ProcessedData"
result_dir = (
    project_root / "result" / "8_real_data" / "2_timelapse_inversion_real_adtlert"
)
if not result_dir.exists():
    raise FileNotFoundError(
        f"Missing time-lapse inversion result directory: {result_dir}"
    )

save_figures = True
figure_dpi = 600

# WRR-style correction parameters.
Tmean_degC = 6.5
ct_percent_per_degC = 2.0
ct_fraction_per_degC = ct_percent_per_degC / 100.0

# A conservative deep anchor: below this depth, temperature relaxes to Tmean.
deep_anchor_depth_m = 5.0
min_correction_factor = 0.25
max_correction_factor = 2.5

# Plot controls.
temp_clim = (-2.0, 8.0)
delta_clim = (-0.25, 0.25)
resistivity_clim = (50.0, 2200.0)
depth_ylim = (-90.0, 5.0)
upper_depth_m = 1.0
x_bin_count = 80

selected_labels = [
    "2022-03-26 00:30",
    "2022-04-20 06:30",
    "2022-05-02 06:30",
    "2022-05-12 18:30",
]


def savefig(fig, filename):
    if save_figures:
        out = result_dir / filename
        fig.savefig(out, dpi=figure_dpi, bbox_inches="tight")
        print("saved:", out)


print("project_root =", project_root)
print("result_dir   =", result_dir)

## 1) Load inversion models, mesh, and sensor metadata


In [ ]:
summary = json.loads((result_dir / "timelapse_real_inversion_summary.json").read_text())
models_raw = np.load(result_dir / "final_models.npy")
coverage_mask = np.load(result_dir / "coverage_mask.npy").astype(bool)

used_lines = (result_dir / "used_data_files.txt").read_text().splitlines()
time_labels = [line.split("	")[0] for line in used_lines]
ert_times = pd.to_datetime(time_labels)
measurement_times_days = np.asarray(
    (ert_times - ert_times[0]).total_seconds() / 86400.0, dtype=float
)
label_to_index = {label: idx for idx, label in enumerate(time_labels)}
selected_indices = [
    label_to_index[label] for label in selected_labels if label in label_to_index
]
if len(selected_indices) != len(selected_labels):
    missing = sorted(set(selected_labels).difference(label_to_index))
    raise ValueError(f"Selected labels not found in used_data_files.txt: {missing}")

with np.load(result_dir / "timelapse_inversion_mesh.npz") as mesh_data:
    nodes = np.asarray(mesh_data["nodes"], dtype=float)
    cells = np.asarray(mesh_data["cells"], dtype=np.int32)
    elec_x = np.asarray(mesh_data["elec_x"], dtype=float)
    elec_z = np.asarray(mesh_data["elec_z"], dtype=float)
    x_nodes = np.asarray(mesh_data["x_nodes"], dtype=float)
    z_top = np.asarray(mesh_data["z_top"], dtype=float)

cell_centers = nodes[cells].mean(axis=1)
cell_x = cell_centers[:, 0]
surface_at_center = np.interp(cell_x, x_nodes, z_top)
cell_depth = np.maximum(surface_at_center - cell_centers[:, 1], 0.0)
visible_cells = (~coverage_mask) & np.isfinite(models_raw[:, 0]) & (cell_depth >= 0.0)

sensor_locations_file = result_dir / "real_sensor_locations_projected.csv"
if sensor_locations_file.exists():
    sensor_locations = pd.read_csv(sensor_locations_file)
else:
    raise FileNotFoundError(
        "Run 4_temperature_and_soil_moisture_QA.ipynb first to create real_sensor_locations_projected.csv"
    )

print("n_times:", models_raw.shape[1])
print("n_cells:", models_raw.shape[0])
print("selected:", [(i, time_labels[i]) for i in selected_indices])
sensor_locations[["ID", "profile_x_m", "relative_elevation_m"]]

## 2) Build temperature profiles at TMC stations


In [ ]:
def read_units_csv(path: Path, time_col: str) -> pd.DataFrame:
    df = pd.read_csv(path, skiprows=[1], low_memory=False)
    df[time_col] = pd.to_datetime(df[time_col])
    df = df.set_index(time_col).sort_index()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.replace(-9999, np.nan)


def interpolate_to_ert_times(df: pd.DataFrame) -> pd.DataFrame:
    merged_index = df.index.union(ert_times)
    return df.reindex(merged_index).interpolate(method="time").reindex(ert_times)


def parse_dtp_depth_m(column: str) -> float | None:
    match = re.match(r"temp_(-?\d+(?:\.\d+)?)$", column)
    if match is None:
        return None
    position_cm = float(match.group(1))
    if position_cm >= 0.0:
        return None
    return abs(position_cm) / 100.0


def parse_teros_depth_m(column: str) -> float | None:
    match = re.match(r"T_(\d+(?:\.\d+)?)m$", column)
    if match is None:
        return None
    return float(match.group(1))


soil_dir = data_dir / "Soil_Moisture_Data"
temp_dir = data_dir / "Soil_Temperature_Data"
soil = {
    path.stem: read_units_csv(path, "DateTime")
    for path in sorted(soil_dir.glob("TMC*.csv"))
}
dtp = {
    path.stem.replace("DTP_", ""): read_units_csv(path, "DateTime")
    for path in sorted(temp_dir.glob("DTP_TMC*.csv"))
}


def station_temperature_profile(station: str) -> tuple[np.ndarray, np.ndarray] | None:
    series_by_depth: list[tuple[float, np.ndarray]] = []

    if station in dtp:
        interp = interpolate_to_ert_times(dtp[station])
        for col in interp.columns:
            depth = parse_dtp_depth_m(col)
            if depth is not None:
                series_by_depth.append((depth, interp[col].to_numpy(dtype=float)))

    if station in soil:
        interp = interpolate_to_ert_times(soil[station])
        for col in interp.columns:
            depth = parse_teros_depth_m(col)
            if depth is not None:
                series_by_depth.append((depth, interp[col].to_numpy(dtype=float)))

    if not series_by_depth:
        return None

    # Add a deep stable-temperature anchor to prevent shallow sensors from controlling deep cells.
    series_by_depth.append(
        (deep_anchor_depth_m, np.full(len(ert_times), Tmean_degC, dtype=float))
    )

    rounded_depths = sorted(set(round(depth, 4) for depth, _ in series_by_depth))
    values = np.full((len(ert_times), len(rounded_depths)), np.nan, dtype=float)
    for depth_idx, rounded_depth in enumerate(rounded_depths):
        stack = [
            values_at_time
            for depth, values_at_time in series_by_depth
            if round(depth, 4) == rounded_depth
        ]
        values[:, depth_idx] = np.nanmean(np.vstack(stack), axis=0)

    # Fill any remaining gaps vertically and then by Tmean if a whole time row is missing.
    depths = np.asarray(rounded_depths, dtype=float)
    for time_idx in range(values.shape[0]):
        row = values[time_idx]
        good = np.isfinite(row)
        if good.sum() >= 2:
            values[time_idx] = np.interp(depths, depths[good], row[good])
        elif good.sum() == 1:
            values[time_idx] = row[good][0]
        else:
            values[time_idx] = Tmean_degC
    return depths, values


station_profiles = {}
for station in sensor_locations["ID"]:
    profile = station_temperature_profile(station)
    if profile is not None:
        station_profiles[station] = profile

print("Temperature profiles available for:", sorted(station_profiles))
for station, (depths, values) in station_profiles.items():
    print(
        station,
        "depth range",
        float(depths.min()),
        "to",
        float(depths.max()),
        "m",
        "n_depths",
        depths.size,
    )

In [ ]:
# Interpolate station temperature profiles to every ERT inversion cell and time.
stations_sorted = [
    station
    for station in sensor_locations.sort_values("profile_x_m")["ID"]
    if station in station_profiles
]
station_x = np.asarray(
    [
        sensor_locations.loc[sensor_locations["ID"].eq(station), "profile_x_m"].iloc[0]
        for station in stations_sorted
    ],
    dtype=float,
)

temperature_field = np.full_like(models_raw, Tmean_degC, dtype=float)
clipped_depth = np.minimum(cell_depth, deep_anchor_depth_m)

for time_idx in range(models_raw.shape[1]):
    station_cell_temps = []
    for station in stations_sorted:
        depths, values = station_profiles[station]
        profile_values = values[time_idx]
        station_cell_temps.append(
            np.interp(
                clipped_depth,
                depths,
                profile_values,
                left=profile_values[0],
                right=Tmean_degC,
            )
        )
    station_cell_temps = np.vstack(station_cell_temps)
    for cell_idx in range(models_raw.shape[0]):
        temperature_field[cell_idx, time_idx] = np.interp(
            cell_x[cell_idx],
            station_x,
            station_cell_temps[:, cell_idx],
            left=station_cell_temps[0, cell_idx],
            right=station_cell_temps[-1, cell_idx],
        )

# Keep masked/outside cells finite for saving, but plotting will use coverage_mask.
print("temperature_field shape:", temperature_field.shape)
print(
    "temperature min/max:",
    float(np.nanmin(temperature_field)),
    float(np.nanmax(temperature_field)),
)

## 3) Apply temperature correction


In [ ]:
correction_factor = 1.0 + ct_fraction_per_degC * (Tmean_degC - temperature_field)
correction_factor = np.clip(
    correction_factor, min_correction_factor, max_correction_factor
)
models_corrected = models_raw * correction_factor

rho0_raw = models_raw[:, 0]
rho0_corrected = models_corrected[:, 0]
delta_raw = (models_raw - rho0_raw[:, None]) / rho0_raw[:, None]
delta_corrected = (models_corrected - rho0_corrected[:, None]) / rho0_corrected[:, None]
delta_temperature_effect = delta_corrected - delta_raw

np.save(result_dir / "temperature_field_sensor_interpolated.npy", temperature_field)
np.save(result_dir / "temperature_correction_factor.npy", correction_factor)
np.save(result_dir / "final_models_temperature_corrected.npy", models_corrected)
np.save(result_dir / "delta_rho_over_rho0_temperature_corrected.npy", delta_corrected)
np.save(
    result_dir / "delta_rho_over_rho0_temperature_effect.npy", delta_temperature_effect
)

correction_summary = {
    "method": "sensor_interpolated_DTP_TEROS_with_deep_Tmean_anchor",
    "Tmean_degC": float(Tmean_degC),
    "ct_percent_per_degC": float(ct_percent_per_degC),
    "deep_anchor_depth_m": float(deep_anchor_depth_m),
    "stations_used": stations_sorted,
    "temperature_min_degC": float(np.nanmin(temperature_field)),
    "temperature_max_degC": float(np.nanmax(temperature_field)),
    "correction_factor_min": float(np.nanmin(correction_factor)),
    "correction_factor_max": float(np.nanmax(correction_factor)),
    "baseline_time": time_labels[0],
    "n_times": int(models_raw.shape[1]),
    "n_cells": int(models_raw.shape[0]),
}
(result_dir / "temperature_correction_summary.json").write_text(
    json.dumps(correction_summary, indent=2), encoding="utf-8"
)
correction_summary

## 4) Plot interpolated temperature fields


In [ ]:
def add_same_height_colorbar(
    fig, ax, mappable, label=None, title=None, size="4%", pad=0.04
):
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=size, pad=pad)
    cbar = fig.colorbar(mappable, cax=cax)
    if label is not None:
        cbar.set_label(label)
    if title is not None:
        cbar.ax.set_title(title, pad=4)
    cbar.ax.tick_params(labelsize=10)
    return cbar


def format_section_axis(ax, *, show_xlabel=True, show_ylabel=True):
    ax.plot(elec_x, elec_z, color="black", lw=1.1, zorder=5)
    ax.set_xlim(float(elec_x.min()), float(elec_x.max()))
    ax.set_ylim(*depth_ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(False)
    if show_xlabel:
        ax.set_xlabel("Distance (m)")
    else:
        ax.tick_params(axis="x", labelbottom=False)
    if show_ylabel:
        ax.set_ylabel("Relative elevation (m)")
    else:
        ax.tick_params(axis="y", labelleft=False)


def plot_cell_model(ax, values, *, cmap, norm, mask=None):
    values = np.asarray(values, dtype=float).ravel()
    visible = np.isfinite(values)
    if mask is not None:
        visible &= ~np.asarray(mask, dtype=bool).ravel()
    pc = PolyCollection(
        nodes[cells][visible],
        array=values[visible],
        cmap=cmap,
        norm=norm,
        edgecolors="none",
    )
    ax.add_collection(pc)
    return pc

In [ ]:
temp_norm = plt.Normalize(vmin=temp_clim[0], vmax=temp_clim[1])
fig, axes = plt.subplots(
    1, len(selected_indices), figsize=(16.2, 4.2), sharex=True, sharey=True
)
axes = np.atleast_1d(axes)
last_pc = None
for col, (ax, idx) in enumerate(zip(axes, selected_indices, strict=True)):
    last_pc = plot_cell_model(
        ax,
        temperature_field[:, idx],
        cmap="coolwarm",
        norm=temp_norm,
        mask=coverage_mask,
    )
    format_section_axis(ax, show_xlabel=True, show_ylabel=(col == 0))
    ax.set_title(time_labels[idx])
if last_pc is not None:
    cbar = fig.colorbar(last_pc, ax=axes, fraction=0.018, pad=0.02)
    cbar.set_label("Temperature (deg C)")
fig.subplots_adjust(wspace=0.08, right=0.92)
savefig(fig, "real_temperature_field_sensor_interpolated.png")
plt.show()

## 5) Corrected relative resistivity change


In [ ]:
delta_norm = TwoSlopeNorm(vmin=delta_clim[0], vcenter=0.0, vmax=delta_clim[1])
fig, axes = plt.subplots(
    2, len(selected_indices), figsize=(16.2, 7.6), sharex=True, sharey=True
)

for col, idx in enumerate(selected_indices):
    pc_raw = plot_cell_model(
        axes[0, col],
        delta_raw[:, idx],
        cmap="RdBu_r",
        norm=delta_norm,
        mask=coverage_mask,
    )
    format_section_axis(axes[0, col], show_xlabel=False, show_ylabel=(col == 0))
    axes[0, col].set_title(time_labels[idx])
    if col == 0:
        axes[0, col].text(
            0.01,
            0.95,
            "Raw",
            transform=axes[0, col].transAxes,
            va="top",
            ha="left",
            fontweight="bold",
        )

    pc_corr = plot_cell_model(
        axes[1, col],
        delta_corrected[:, idx],
        cmap="RdBu_r",
        norm=delta_norm,
        mask=coverage_mask,
    )
    format_section_axis(axes[1, col], show_xlabel=True, show_ylabel=(col == 0))
    if col == 0:
        axes[1, col].text(
            0.01,
            0.95,
            "Temperature corrected",
            transform=axes[1, col].transAxes,
            va="top",
            ha="left",
            fontweight="bold",
        )

cbar = fig.colorbar(pc_corr, ax=axes, fraction=0.016, pad=0.02)
cbar.set_label("Delta rho / rho0")
fig.subplots_adjust(wspace=0.08, hspace=0.10, right=0.92)
savefig(fig, "real_delta_rho_over_rho0_raw_vs_temperature_corrected.png")
plt.show()

In [ ]:
# Plot the isolated temperature-correction contribution.
effect_norm = TwoSlopeNorm(vmin=-0.15, vcenter=0.0, vmax=0.15)
fig, axes = plt.subplots(
    1, len(selected_indices), figsize=(16.2, 4.2), sharex=True, sharey=True
)
axes = np.atleast_1d(axes)
last_pc = None
for col, (ax, idx) in enumerate(zip(axes, selected_indices, strict=True)):
    last_pc = plot_cell_model(
        ax,
        delta_temperature_effect[:, idx],
        cmap="BrBG",
        norm=effect_norm,
        mask=coverage_mask,
    )
    format_section_axis(ax, show_xlabel=True, show_ylabel=(col == 0))
    ax.set_title(time_labels[idx])
if last_pc is not None:
    cbar = fig.colorbar(last_pc, ax=axes, fraction=0.018, pad=0.02)
    cbar.set_label("Corrected minus raw Delta rho / rho0")
fig.subplots_adjust(wspace=0.08, right=0.92)
savefig(fig, "real_temperature_correction_effect_on_delta_rho.png")
plt.show()

## 6) Uppermost 1 m corrected response


In [ ]:
x_edges = np.linspace(float(elec_x.min()), float(elec_x.max()), x_bin_count + 1)
x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
upper_mask = visible_cells & (cell_depth >= 0.0) & (cell_depth <= upper_depth_m)


def upper_profile(delta_values):
    profile = np.full((models_raw.shape[1], x_centers.size), np.nan, dtype=float)
    for bin_idx in range(x_centers.size):
        in_bin = (
            upper_mask & (cell_x >= x_edges[bin_idx]) & (cell_x < x_edges[bin_idx + 1])
        )
        if np.any(in_bin):
            profile[:, bin_idx] = np.nanmean(delta_values[in_bin, :], axis=0)
    return profile


upper_raw = upper_profile(delta_raw)
upper_corrected = upper_profile(delta_corrected)

time_edges = np.empty(measurement_times_days.size + 1, dtype=float)
time_edges[1:-1] = 0.5 * (measurement_times_days[:-1] + measurement_times_days[1:])
time_edges[0] = measurement_times_days[0] - (time_edges[1] - measurement_times_days[0])
time_edges[-1] = measurement_times_days[-1] + (
    measurement_times_days[-1] - time_edges[-2]
)

fig, axes = plt.subplots(2, 1, figsize=(12.0, 7.0), sharex=True, sharey=True)
mesh0 = axes[0].pcolormesh(
    x_edges, time_edges, upper_raw, cmap="RdBu_r", norm=delta_norm, shading="auto"
)
axes[0].set_title(f"Raw uppermost {upper_depth_m:g} m Delta rho / rho0")
axes[0].set_ylabel("Days since baseline")
axes[0].grid(False)

mesh1 = axes[1].pcolormesh(
    x_edges, time_edges, upper_corrected, cmap="RdBu_r", norm=delta_norm, shading="auto"
)
axes[1].set_title(
    f"Temperature-corrected uppermost {upper_depth_m:g} m Delta rho / rho0"
)
axes[1].set_ylabel("Days since baseline")
axes[1].set_xlabel("Distance (m)")
axes[1].grid(False)

cbar = fig.colorbar(mesh1, ax=axes, fraction=0.020, pad=0.015)
cbar.set_label("Delta rho / rho0")
fig.tight_layout()
savefig(fig, "real_upper1m_delta_rho_temperature_corrected.png")
plt.show()

## Output files

The following arrays are saved in the real-data result directory:

- `temperature_field_sensor_interpolated.npy`
- `temperature_correction_factor.npy`
- `final_models_temperature_corrected.npy`
- `delta_rho_over_rho0_temperature_corrected.npy`
- `delta_rho_over_rho0_temperature_effect.npy`
- `temperature_correction_summary.json`
